# Sliced Normal

This is the original repository workflow using the cached implementation in
`sliced_normals/`: data → FMLE → fit → approximate likelihood → MCMC → plots.

Run the first code cell in your current kernel to create `.venv`, install the
requirements, and register **Python (Sliced Normal)**. Select that kernel, then
run all cells. Python 3.11 is required for this setup cell. It uses standard
library modules and does not clone or delete a repository.

The sampler retains unbounded support while fitting normalizes on a box.
Sampling plots are exploratory and do not establish convergence.

In [ ]:
# Run this cell first. It does not require NumPy, pandas, or the fitting packages.
import os
import sys
import json
import shlex
import shutil
import hashlib
import subprocess
from pathlib import Path

BASE_PYTHON = None  # Optional: '/absolute/path/to/python3.11'
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'sliced_normals' / 'optimization.py').exists()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from the repository root or one of its subfolders.')
ENV_DIR = ROOT / '.venv'
ENV_PYTHON = ENV_DIR / ('Scripts/python.exe' if os.name == 'nt' else 'bin/python')
REQUIREMENTS = ROOT / 'requirements.txt'
KERNEL_NAME = 'sliced-normal'
KERNEL_LABEL = 'Python (Sliced Normal)'

def run_setup_command(args):
    args = [str(arg) for arg in args]
    print('>', shlex.join(args), flush=True)
    with subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        status = process.wait()
    if status:
        raise RuntimeError(f'Setup command failed with exit code {status}. See the output above.')

def python_version(executable):
    try:
        result = subprocess.run(
            [str(executable), '-c', 'import sys,json; print(json.dumps(list(sys.version_info[:2])))'],
            check=True, capture_output=True, text=True)
        return tuple(json.loads(result.stdout))
    except (OSError, subprocess.SubprocessError, ValueError):
        return None

if not REQUIREMENTS.is_file():
    raise RuntimeError(f'Missing dependency file: {REQUIREMENTS}')
if not ENV_PYTHON.is_file():
    candidates = ([BASE_PYTHON] if BASE_PYTHON else
                  [shutil.which('python3.11'), sys.executable, shutil.which('python3')])
    base = next((candidate for candidate in candidates
                 if candidate and python_version(candidate) == (3, 11)), None)
    if base is None:
        raise RuntimeError('Install Python 3.11, set BASE_PYTHON above to its executable, and rerun this cell.')
    run_setup_command([base, '-m', 'venv', ENV_DIR])
if python_version(ENV_PYTHON) != (3, 11):
    raise RuntimeError(f'{ENV_DIR} is not a working Python 3.11 environment. '
                       'Choose a different ENV_DIR or recreate that environment, then rerun setup.')

run_setup_command([ENV_PYTHON, '-m', 'pip', '--require-virtualenv', 'install', '-r', REQUIREMENTS])
run_setup_command([ENV_PYTHON, '-m', 'pip', '--require-virtualenv', 'install', '-e', ROOT / 'sliced_normals'])
run_setup_command([ENV_PYTHON, '-m', 'pip', 'check'])
run_setup_command([
    ENV_PYTHON, '-c',
    'import sys; sys.path.insert(0, sys.argv[1]); '
    'import numpy,pandas,scipy,autograd,pymanopt,emcee,matplotlib,threadpoolctl,ipykernel; '
    'import sliced_normals; '
    'print("All notebook and fitter imports passed.")', str(ROOT)])
run_setup_command([ENV_PYTHON, '-m', 'ipykernel', 'install', '--user',
                   '--name', KERNEL_NAME, '--display-name', KERNEL_LABEL])

print('\nEnvironment:', ENV_DIR)
print('Current kernel:', sys.executable)
if Path(sys.prefix).resolve() == ENV_DIR.resolve():
    print('The notebook is already using the new environment. Continue with the next cell.')
else:
    print(f'NEXT: select "{KERNEL_LABEL}" in the notebook kernel selector, then run all cells.')
    print('If it is not listed, refresh the kernel list or choose this interpreter:', ENV_PYTHON)
if os.name != 'nt':
    print('Optional terminal activation: source ' + shlex.quote(str(ENV_DIR / 'bin/activate')))
else:
    print('Optional PowerShell activation: & "' + str(ENV_DIR / 'Scripts/Activate.ps1') + '"')

In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p/'sliced_normals/optimization.py').exists()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from the repository root or Demo folder.')
if Path(sys.prefix).resolve() != (ROOT/'.venv').resolve():
    raise RuntimeError('Select the Python (Sliced Normal) kernel created by the setup cell, then run all cells.')
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pymanopt.manifolds import SymmetricPositiveDefinite
from sliced_normals import (get_FMLE, estimate_optimal_B, evaluate_true_log_likelihood,
                            run_mcmc_sampler, get_F_and_Random_Samples, CachedObjective)
print('Python:', sys.executable)


In [ ]:
DEGREE = 2
GRID = 5000
MAX_ITERATIONS = 2000
GRADIENT_TOLERANCE = 5e-4
EVALUATION_GRID = 50000
STEPS = 2000
BURNIN = 300
THIN = 5
SEED = 42


## Data and FMLE

In [ ]:
data = pd.read_csv(ROOT/'sliced_normals/Sample_Data.csv')[['X_1','X_2','X_3']]
np.random.seed(SEED)
B_fmle = get_FMLE(data, DEGREE)
print('Observations:', len(data))
sns.pairplot(data, plot_kws={'s': 5})
plt.show()


## Maximum-likelihood fitting
The default objective retains the partition floor and Pymanopt steepest-descent steps. Stopping is checked at the same matrix the fitter returns; this cell independently checks its gradient. An iteration/time limit or tiny-step stop can still leave the requested gradient tolerance unmet.

In [ ]:
np.random.seed(SEED)
B_opt = estimate_optimal_B(data, degree=DEGREE, n_grid=GRID,
                           max_iterations=MAX_ITERATIONS,
                           min_gradient_norm=GRADIENT_TOLERANCE, verbosity=1)
np.random.seed(SEED)
Z, Z_grid, volume = get_F_and_Random_Samples(data, DEGREE, GRID)
objective = CachedObjective(Z.to_numpy(), Z_grid.to_numpy(), volume)
manifold = SymmetricPositiveDefinite(len(B_opt))
rgrad = manifold.euclidean_to_riemannian_gradient(B_opt, objective.gradient(B_opt))
final_gradient_norm = float(manifold.norm(B_opt, rgrad))
print('Final objective:', objective.cost(B_opt))
print('Final gradient norm:', final_gradient_norm)
print('Requested gradient tolerance met:', final_gradient_norm < GRADIENT_TOLERANCE)


## Independent normalization check
These are approximate in-sample mean log likelihoods. Both models use identical fresh integration points on the same data-derived support. This is not held-out validation.

In [ ]:
for name, B in [('FMLE', B_fmle), ('Fitted model', B_opt)]:
    np.random.seed(SEED + 1000)
    value = evaluate_true_log_likelihood(data, B, DEGREE, n_grid=EVALUATION_GRID)
    print(name, 'approximate mean log likelihood:', float(value))


## MCMC sampling
STEPS counts steps per walker. Check chain mixing, effective sample size, and mode coverage separately.

In [ ]:
np.random.seed(SEED + 2000)
samples_opt = run_mcmc_sampler(data, B_opt, d=DEGREE, num_samples=STEPS,
                               burnin=BURNIN, thin=THIN)
np.random.seed(SEED + 2001)
samples_fmle = run_mcmc_sampler(data, B_fmle, d=DEGREE, num_samples=STEPS,
                                burnin=BURNIN, thin=THIN)
print('Retained rows:', len(samples_opt), len(samples_fmle))


## Pairplots
Use common coordinate limits to compare observed data and retained samples.

In [ ]:
frames = [('Data', data), ('Fitted model samples', samples_opt), ('FMLE samples', samples_fmle)]
limits = {column: (min(frame[column].min() for _, frame in frames),
                   max(frame[column].max() for _, frame in frames)) for column in data.columns}
for name, frame in frames:
    plot = sns.pairplot(frame, plot_kws={'s': 5})
    plot.fig.suptitle(name, y=1.02)
    for i, y in enumerate(data.columns):
        for j, x in enumerate(data.columns):
            plot.axes[i,j].set_xlim(limits[x])
            if i != j:
                plot.axes[i,j].set_ylim(limits[y])
    plt.show()
